In [1]:
!pip install -q wandb xgboost scikit-learn numpy pandas

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shurpalitanmayi2000 (shurpalitanmayi2000-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Download dataset
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data -qq

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset
data = np.loadtxt(
    "./dermatology.data",
    delimiter=",",
    converters={
        33: lambda x: int(x == b'?'),
        34: lambda x: int(x) - 1
    }
)

X = data[:, :33]
y = data[:, 34]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training samples:", X_train.shape)
print("Test samples:", X_test.shape)

Training samples: (256, 33)
Test samples: (110, 33)


In [5]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

run = wandb.init(project="Lab1-visualize-models", name="baseline-model")

xg_train = xgb.DMatrix(X_train, label=y_train)
xg_test = xgb.DMatrix(X_test, label=y_test)

params = {
    "objective": "multi:softmax",
    "eta": 0.1,
    "max_depth": 6,
    "nthread": 4,
    "num_class": 6
}

wandb.config.update(params)

model = xgb.train(
    params,
    xg_train,
    num_boost_round=20,
    evals=[(xg_train, "train"), (xg_test, "test")],
    callbacks=[wandb.xgboost.WandbCallback()]
)

pred = model.predict(xg_test)

accuracy = accuracy_score(y_test, pred)
print("Baseline Accuracy:", accuracy)

wandb.log({"accuracy": accuracy})

[0]	train-mlogloss:1.45254	test-mlogloss:1.46694
[1]	train-mlogloss:1.27203	test-mlogloss:1.30080
[2]	train-mlogloss:1.12344	test-mlogloss:1.17017
[3]	train-mlogloss:0.99975	test-mlogloss:1.06118
[4]	train-mlogloss:0.89462	test-mlogloss:0.96987
[5]	train-mlogloss:0.80471	test-mlogloss:0.89142
[6]	train-mlogloss:0.72497	test-mlogloss:0.81871
[7]	train-mlogloss:0.65499	test-mlogloss:0.75528
[8]	train-mlogloss:0.59361	test-mlogloss:0.69982
[9]	train-mlogloss:0.53906	test-mlogloss:0.65050
[10]	train-mlogloss:0.49053	test-mlogloss:0.60679
[11]	train-mlogloss:0.44729	test-mlogloss:0.56685
[12]	train-mlogloss:0.40854	test-mlogloss:0.53076
[13]	train-mlogloss:0.37354	test-mlogloss:0.49837
[14]	train-mlogloss:0.34234	test-mlogloss:0.46944
[15]	train-mlogloss:0.31404	test-mlogloss:0.44327
[16]	train-mlogloss:0.28859	test-mlogloss:0.41956
[17]	train-mlogloss:0.26530	test-mlogloss:0.39831
[18]	train-mlogloss:0.24454	test-mlogloss:0.37911
[19]	train-mlogloss:0.22576	test-mlogloss:0.36220
Baseline A

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, pred, average="weighted", zero_division=0)
recall = recall_score(y_test, pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, pred, average="weighted", zero_division=0)
error_rate = 1 - accuracy

wandb.log({
    "precision_weighted": precision,
    "recall_weighted": recall,
    "f1_weighted": f1,
    "error_rate": error_rate
})

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Error Rate:", error_rate)

Precision: 0.9368724004753416
Recall: 0.9363636363636364
F1 Score: 0.935687753107108
Error Rate: 0.0636363636363636


In [7]:
wandb.sklearn.plot_confusion_matrix(
    y_test,
    pred,
    labels=[0., 1., 2., 3., 4., 5.]
)

In [8]:
learning_rates = [0.01, 0.1, 0.3]
max_depths = [3, 5, 7]

for eta in learning_rates:
    for depth in max_depths:
        run = wandb.init(
            project="Lab1-visualize-models",
            name=f"xgb_eta_{eta}_depth_{depth}",
            reinit=True
        )

        params = {
            "objective": "multi:softmax",
            "eta": eta,
            "max_depth": depth,
            "nthread": 4,
            "num_class": 6
        }

        wandb.config.update(params)

        model = xgb.train(
            params,
            xg_train,
            num_boost_round=30,
            evals=[(xg_train, "train"), (xg_test, "test")],
            callbacks=[wandb.xgboost.WandbCallback()]
        )

        pred = model.predict(xg_test)

        accuracy = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

        wandb.log({
            "accuracy": accuracy,
            "f1_weighted": f1
        })

        print(f"eta={eta}, depth={depth}, accuracy={accuracy:.4f}")

        run.finish()

accuracy,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
error_rate,▁
f1_weighted,▁
precision_weighted,▁
recall_weighted,▁
test-mlogloss,█▇▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
train-mlogloss,█▇▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
accuracy,0.93636
epoch,19
error_rate,0.06364


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[0]	train-mlogloss:1.66364	test-mlogloss:1.66155
[1]	train-mlogloss:1.64054	test-mlogloss:1.64020
[2]	train-mlogloss:1.61802	test-mlogloss:1.61939
[3]	train-mlogloss:1.59606	test-mlogloss:1.59910
[4]	train-mlogloss:1.57463	test-mlogloss:1.57931
[5]	train-mlogloss:1.55371	test-mlogloss:1.56000
[6]	train-mlogloss:1.53329	test-mlogloss:1.54114
[7]	train-mlogloss:1.51333	test-mlogloss:1.52273
[8]	train-mlogloss:1.49382	test-mlogloss:1.50473
[9]	train-mlogloss:1.47474	test-mlogloss:1.48713
[10]	train-mlogloss:1.45608	test-mlogloss:1.46993
[11]	train-mlogloss:1.43782	test-mlogloss:1.45310
[12]	train-mlogloss:1.41995	test-mlogloss:1.43662
[13]	train-mlogloss:1.40244	test-mlogloss:1.42050
[14]	train-mlogloss:1.38530	test-mlogloss:1.40471
[15]	train-mlogloss:1.36850	test-mlogloss:1.38924
[16]	train-mlogloss:1.35197	test-mlogloss:1.37423
[17]	train-mlogloss:1.33576	test-mlogloss:1.35953
[18]	train-mlogloss:1.31987	test-mlogloss:1.34511
[19]	train-mlogloss:1.30428	test-mlogloss:1.33087
[20]	train

accuracy,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
f1_weighted,▁
test-mlogloss,██▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
train-mlogloss,██▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
accuracy,0.8
epoch,29
f1_weighted,0.77211


[0]	train-mlogloss:1.66326	test-mlogloss:1.66123
[1]	train-mlogloss:1.63979	test-mlogloss:1.63952
[2]	train-mlogloss:1.61691	test-mlogloss:1.61836
[3]	train-mlogloss:1.59460	test-mlogloss:1.59778
[4]	train-mlogloss:1.57282	test-mlogloss:1.57766
[5]	train-mlogloss:1.55157	test-mlogloss:1.55803
[6]	train-mlogloss:1.53081	test-mlogloss:1.53890
[7]	train-mlogloss:1.51052	test-mlogloss:1.52018
[8]	train-mlogloss:1.49069	test-mlogloss:1.50188
[9]	train-mlogloss:1.47130	test-mlogloss:1.48404
[10]	train-mlogloss:1.45234	test-mlogloss:1.46655
[11]	train-mlogloss:1.43378	test-mlogloss:1.44944
[12]	train-mlogloss:1.41561	test-mlogloss:1.43274
[13]	train-mlogloss:1.39765	test-mlogloss:1.41654
[14]	train-mlogloss:1.38005	test-mlogloss:1.40067
[15]	train-mlogloss:1.36282	test-mlogloss:1.38518
[16]	train-mlogloss:1.34592	test-mlogloss:1.36995
[17]	train-mlogloss:1.32936	test-mlogloss:1.35503
[18]	train-mlogloss:1.31298	test-mlogloss:1.34055
[19]	train-mlogloss:1.29692	test-mlogloss:1.32631
[20]	train

...

accuracy,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
f1_weighted,▁
test-mlogloss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train-mlogloss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.93636
epoch,29
f1_weighted,0.93643


[0]	train-mlogloss:1.03432	test-mlogloss:1.08411
[1]	train-mlogloss:0.73006	test-mlogloss:0.81738
[2]	train-mlogloss:0.53780	test-mlogloss:0.64627
[3]	train-mlogloss:0.40506	test-mlogloss:0.52380
[4]	train-mlogloss:0.31129	test-mlogloss:0.43551
[5]	train-mlogloss:0.24248	test-mlogloss:0.37178
[6]	train-mlogloss:0.19159	test-mlogloss:0.32586
[7]	train-mlogloss:0.15243	test-mlogloss:0.29060
[8]	train-mlogloss:0.12355	test-mlogloss:0.26679
[9]	train-mlogloss:0.10094	test-mlogloss:0.24794
[10]	train-mlogloss:0.08359	test-mlogloss:0.23765
[11]	train-mlogloss:0.07023	test-mlogloss:0.22457
[12]	train-mlogloss:0.06026	test-mlogloss:0.21435
[13]	train-mlogloss:0.05187	test-mlogloss:0.20544
[14]	train-mlogloss:0.04521	test-mlogloss:0.19899
[15]	train-mlogloss:0.04023	test-mlogloss:0.19664
[16]	train-mlogloss:0.03595	test-mlogloss:0.19435
[17]	train-mlogloss:0.03258	test-mlogloss:0.19135
[18]	train-mlogloss:0.02980	test-mlogloss:0.18804
[19]	train-mlogloss:0.02739	test-mlogloss:0.18611
[20]	train

accuracy,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
f1_weighted,▁
test-mlogloss,█▆▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train-mlogloss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.94545
epoch,29
f1_weighted,0.94616


[0]	train-mlogloss:1.03235	test-mlogloss:1.08346
[1]	train-mlogloss:0.72805	test-mlogloss:0.81706
[2]	train-mlogloss:0.53375	test-mlogloss:0.64474
[3]	train-mlogloss:0.40051	test-mlogloss:0.52476
[4]	train-mlogloss:0.30505	test-mlogloss:0.43681
[5]	train-mlogloss:0.23580	test-mlogloss:0.37357
[6]	train-mlogloss:0.18529	test-mlogloss:0.32762
[7]	train-mlogloss:0.14667	test-mlogloss:0.29268
[8]	train-mlogloss:0.11764	test-mlogloss:0.26763
[9]	train-mlogloss:0.09573	test-mlogloss:0.25100
[10]	train-mlogloss:0.07853	test-mlogloss:0.23678
[11]	train-mlogloss:0.06572	test-mlogloss:0.22566
[12]	train-mlogloss:0.05539	test-mlogloss:0.22340
[13]	train-mlogloss:0.04811	test-mlogloss:0.21859
[14]	train-mlogloss:0.04208	test-mlogloss:0.21445
[15]	train-mlogloss:0.03709	test-mlogloss:0.21094
[16]	train-mlogloss:0.03322	test-mlogloss:0.20894
[17]	train-mlogloss:0.02974	test-mlogloss:0.20428
[18]	train-mlogloss:0.02727	test-mlogloss:0.20193
[19]	train-mlogloss:0.02517	test-mlogloss:0.20057
[20]	train

accuracy,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
f1_weighted,▁
test-mlogloss,█▆▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train-mlogloss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.93636
epoch,29
f1_weighted,0.93722


In [9]:
import os

model_path = "xgboost_model.json"
model.save_model(model_path)

artifact = wandb.Artifact(
    name="xgboost-dermatology-model",
    type="model",
    description="Final trained XGBoost model"
)

artifact.add_file(model_path)

run = wandb.init(project="Lab1-visualize-models", name="model-artifact", reinit=True)
run.log_artifact(artifact)
run.finish()

print("Model saved and logged to W&B")

Model saved and logged to W&B


## Key Takeaways

- successfully tracked machine learning experiments using Weights & Biases.

- Multiple hyperparameter combinations were tested and compared.

- Metrics like accuracy, precision, recall, and F1-score were logged.

- Model artifacts were saved for reproducibility and versioning.

## Future Improvements
- Use W&B Sweeps for automated hyperparameter tuning

- Add cross-validation

- Deploy the model using a REST API